# 01 - Setup Credentials
Validate AWS SSO login, confirm SDK versions, and initialize config.json for downstream notebooks.

## Step 1: Validate AWS Credentials

Ensure you have run `aws sso login --profile bedrock` before starting.

In [ ]:
import boto3
import json
from pathlib import Path

# Validate credentials by calling STS
sts = boto3.client("sts")
try:
    identity = sts.get_caller_identity()
    account_id = identity["Account"]
    arn = identity["Arn"]
    print(f"\u2713 Authenticated successfully")
    print(f"  Account: {account_id}")
    print(f"  ARN: {arn}")
except Exception as e:
    print(f"\u2717 Authentication failed: {e}")
    print("  Run: aws sso login --profile bedrock")
    raise SystemExit("Fix credentials before continuing.")

## Step 2: Validate SDK Versions

Confirm required packages are installed at compatible versions.

In [ ]:
import importlib.metadata

required_packages = [
    ("boto3", "1.34"),
    ("strands-agents", "1.39"),
    ("bedrock-agentcore", "1.9"),
    ("ag-ui-protocol", "0.1.18"),
    ("ag-ui-strands", "0.1.8"),
]

all_ok = True
for package_name, min_version in required_packages:
    try:
        version = importlib.metadata.version(package_name)
        status = "\u2713" if version >= min_version else "\u26a0\ufe0f"
        print(f"  {status} {package_name}: {version} (min: {min_version})")
    except importlib.metadata.PackageNotFoundError:
        print(f"  \u2717 {package_name}: NOT INSTALLED (need >= {min_version})")
        all_ok = False

if all_ok:
    print("\n\u2713 All required packages available")
else:
    print("\n\u2717 Missing packages. Run: pip install -r requirements.txt")

## Step 3: Verify Bedrock Model Access

Confirm we can invoke Claude Sonnet and Titan Embeddings.

In [ ]:
region = "us-east-1"
bedrock_runtime = boto3.client("bedrock-runtime", region_name=region)

# Test Nova Pro access
model_id = "amazon.nova-pro-v1:0"
try:
    response = bedrock_runtime.converse(
        modelId=model_id,
        messages=[{"role": "user", "content": [{"text": "Say hello in one word."}]}],
        inferenceConfig={"maxTokens": 10}
    )
    output_text = response["output"]["message"]["content"][0]["text"]
    print(f"\u2713 Nova Pro accessible: '{output_text}'")
except Exception as e:
    print(f"\u2717 Nova Pro not accessible: {e}")
    print("  Ensure model access is enabled in Bedrock console.")

In [ ]:
# Test Titan Embeddings V2 access
embedding_model_id = "amazon.titan-embed-text-v2:0"
try:
    response = bedrock_runtime.invoke_model(
        modelId=embedding_model_id,
        body=json.dumps({
            "inputText": "test embedding",
            "dimensions": 1024,
            "normalize": True
        })
    )
    result = json.loads(response["body"].read())
    embedding_dim = len(result["embedding"])
    print(f"\u2713 Titan Embeddings V2 accessible: {embedding_dim} dimensions")
    assert embedding_dim == 1024, f"Expected 1024, got {embedding_dim}"
except Exception as e:
    print(f"\u2717 Titan Embeddings V2 not accessible: {e}")

## Step 4: Check AgentCore CLI

Verify the AgentCore CLI is installed and accessible.

In [ ]:
import subprocess

try:
    result = subprocess.run(
        ["agentcore", "--version"],
        capture_output=True, text=True, timeout=10
    )
    if result.returncode == 0:
        print(f"\u2713 AgentCore CLI: {result.stdout.strip()}")
    else:
        print(f"\u2717 AgentCore CLI error: {result.stderr.strip()}")
except FileNotFoundError:
    print("\u2717 AgentCore CLI not found. Install: npm i -g @aws/agentcore")
except Exception as e:
    print(f"\u2717 AgentCore CLI check failed: {e}")

## Step 5: Initialize config.json

Set account info and region in the shared config file.

In [ ]:
config_path = Path("config.json")
config = json.loads(config_path.read_text())

# Update with discovered values
config["ACCOUNT_ID"] = account_id
config["REGION"] = region

config_path.write_text(json.dumps(config, indent=2))
print(f"\u2713 config.json updated")
print(f"  ACCOUNT_ID: {config['ACCOUNT_ID']}")
print(f"  REGION: {config['REGION']}")
print(f"  PROJECT_PREFIX: {config['PROJECT_PREFIX']}")

## Step 6: Load CDK Stack Outputs into config.json

Reads deployed stack outputs from CloudFormation and writes them to `config.json`.
The remaining notebooks depend on these values — run this cell after each `cdk deploy`.

Required stacks: `OrgIQ-Foundation`, `OrgIQ-Aurora`, `OrgIQ-KnowledgeBase`  
Optional stacks: `OrgIQ-Guardrails` (needed for notebooks 05+)

In [ ]:
cfn = boto3.client("cloudformation", region_name=region)

# Stack name → { CloudFormation OutputKey: config.json key }
STACK_MAP = {
    "OrgIQ-Foundation": {
        "DocumentsBucketName": "DOCUMENTS_BUCKET",
        "ArtifactsBucketName": "ARTIFACTS_BUCKET",
        "ScoresTableName":     "SCORES_TABLE",
        "SessionsTableName":   "SESSIONS_TABLE",
        "AgentBaseRoleArn":    "AGENT_BASE_ROLE_ARN",
    },
    "OrgIQ-Aurora": {
        "ClusterArn":           "AURORA_CLUSTER_ARN",
        "CredentialsSecretArn": "AURORA_SECRET_ARN",
        "ClusterEndpoint":      "AURORA_ENDPOINT",
        "DbName":               "AURORA_DB_NAME",
    },
    "OrgIQ-KnowledgeBase": {
        "KnowledgeBaseId":  "KB_ID",
        "KnowledgeBaseArn": "KB_ARN",
    },
    "OrgIQ-Guardrails": {
        "GuardrailId":      "GUARDRAIL_ID",
        "GuardrailArn":     "GUARDRAIL_ARN",
        "GuardrailVersion": "GUARDRAIL_VERSION",
    },
}

missing_required = []

for stack_name, key_map in STACK_MAP.items():
    required = stack_name in ("OrgIQ-Foundation", "OrgIQ-Aurora", "OrgIQ-KnowledgeBase")
    try:
        resp = cfn.describe_stacks(StackName=stack_name)
        outputs = {o["OutputKey"]: o["OutputValue"]
                   for o in resp["Stacks"][0].get("Outputs", [])}
        status = resp["Stacks"][0]["StackStatus"]

        if status not in ("CREATE_COMPLETE", "UPDATE_COMPLETE"):
            symbol = "⚠" if required else "·"
            print(f"  {symbol} {stack_name}: status={status} (not ready)")
            if required:
                missing_required.append(stack_name)
            continue

        for cfn_key, config_key in key_map.items():
            if cfn_key in outputs:
                config[config_key] = outputs[cfn_key]

        print(f"  ✓ {stack_name}: {len(key_map)} values loaded")

    except cfn.exceptions.ClientError as e:
        if "does not exist" in str(e):
            symbol = "✗" if required else "·"
            print(f"  {symbol} {stack_name}: not deployed")
            if required:
                missing_required.append(stack_name)
        else:
            raise

config_path.write_text(json.dumps(config, indent=2))
print(f"\n✓ config.json updated ({len(config)} keys)")

if missing_required:
    print(f"\n⚠ Required stacks not yet deployed: {missing_required}")
    print("  Run: cd infra && cdk deploy " + " ".join(missing_required))
else:
    print(f"  AURORA_CLUSTER_ARN : {config.get('AURORA_CLUSTER_ARN', '(missing)')}")
    print(f"  KB_ID              : {config.get('KB_ID', '(missing)')}")
    print(f"  DOCUMENTS_BUCKET   : {config.get('DOCUMENTS_BUCKET', '(missing)')}")

## Step 7: Initialize Resource Tracker

Set up the resource tracker for safe cleanup later.

In [ ]:
from cleanup_utils import ResourceTracker

tracker = ResourceTracker()
tracker.set_account_info(account_id=account_id, region=region)
print("\u2713 Resource tracker initialized")
tracker.summary()

## Step 8: Verify Available AWS Services

Quick check that key services are reachable.

In [ ]:
services_to_check = [
    ("neptune-graph", "Neptune Analytics"),
    ("bedrock", "Bedrock"),
    ("bedrock-agent", "Bedrock Agent"),
    ("opensearchserverless", "OpenSearch Serverless"),
    ("s3", "S3"),
    ("dynamodb", "DynamoDB"),
]

for service_name, display_name in services_to_check:
    try:
        client = boto3.client(service_name, region_name=region)
        print(f"  \u2713 {display_name} client created")
    except Exception as e:
        print(f"  \u2717 {display_name} unavailable: {e}")

print("\n\u2713 Service connectivity verified")

## Summary

- [x] AWS credentials validated (STS get-caller-identity)
- [x] SDK versions confirmed
- [x] Bedrock model access verified (Claude Sonnet + Titan Embeddings)
- [x] AgentCore CLI checked
- [x] config.json initialized with account info
- [x] Resource tracker ready
- [x] AWS services reachable

**Next:** [02-neptune-graph.ipynb](02-neptune-graph.ipynb) — Create Neptune Analytics graph with org schema